In [45]:
import pandas as pd
df = pd.read_csv("../data/processed/processed_data.csv")
df.head(10)

,actual_price,discount_percentage,rating,rating_count,review_title,review_content,rating_area
0,1099.0,64.0,4.2,24269,"Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,1
1,349.0,43.0,4.0,43994,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,0
2,1899.0,90.0,3.9,7928,"Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",0
3,699.0,53.0,4.2,94363,"Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",1
4,399.0,61.0,4.2,16905,"As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",1
5,1000.0,85.0,3.9,24871,"It's pretty good,Average quality,very good and...","It's a good product.,Like,Very good item stron...",0
6,499.0,65.0,4.1,15188,"Long durable.,good,Does not charge Lenovo m8 t...",Build quality is good and it is comes with 2 y...,1
7,299.0,23.0,4.3,30411,"Worth for money - suitable for Android auto,Go...",Worth for money - suitable for Android auto......,2
8,999.0,50.0,4.2,179691,Works on linux for me. Get the model with ante...,I use this to connect an old PC to internet. I...,1
9,299.0,33.0,4.0,43994,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,0


In [46]:
# 피처 구분 - ML X/y, LLM 입력
X = df[["actual_price", "discount_percentage", "rating_count"]]
y = df["rating_area"]
llm_input = df[["review_title", "review_content"]]

In [47]:
# stratify=y 추가함
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [48]:
# 모델링
# 파라미터 모두 기본값
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score

In [49]:
# =========================
# 실험 0. 기본 성능 (파라미터/스케일링 x)
# =========================
exp0_models = {
  'lr' : LogisticRegression(random_state=42, max_iter=1000),
  'rf' : RandomForestClassifier(random_state=42),
  'gb' : GradientBoostingClassifier(random_state=42)
}

for name, model in exp0_models.items():
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='weighted')
  print(f"<{name}> Accuracy score : {acc:.4f}")
  print(f"<{name}> F1 score : {f1:.4f}")
  print()

<lr> Accuracy score : 0.4608
<lr> F1 score : 0.4233

<rf> Accuracy score : 0.5529
<rf> F1 score : 0.5515

<gb> Accuracy score : 0.5085
<gb> F1 score : 0.5036



In [50]:
# =========================
# 실험 1. 세 모델 모두 스케일링 적용 >> rf, gb는 실험0과 차이없음을 확인
# =========================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

exp1_models = {
  'lr' : LogisticRegression(random_state=42, max_iter=1000),
  'rf' : RandomForestClassifier(random_state=42),
  'gb' : GradientBoostingClassifier(random_state=42)
}

for name, model in exp1_models.items():
  model.fit(X_train_scaled, y_train)

  y_pred = model.predict(X_test_scaled)
  acc = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='weighted')
  print(f"<{name}> Accuracy score : {acc:.4f}")
  print(f"<{name}> F1 score : {f1:.4f}")
  print()

<lr> Accuracy score : 0.4573
<lr> F1 score : 0.4197

<rf> Accuracy score : 0.5529
<rf> F1 score : 0.5518

<gb> Accuracy score : 0.5119
<gb> F1 score : 0.5072



In [51]:
# =========================
# 실험 2. lr만 스케일링 + 세 모델 모두 GridSearchCV
# =========================
from sklearn.model_selection import GridSearchCV

lr_params = {
  'C'           : [0.01, 0.1, 1, 10, 100],
  'penalty'     : ['l2'],
  'solver'      : ['lbfgs', 'newton-cg', 'sag', 'saga'],
  'max_iter'    : [1000],
}

rf_params = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features'     : ['sqrt', 'log2', None],
}

gb_params = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth'    : [3, 4, 5],
    'subsample'    : [0.7, 0.8, 1.0],
}

exp2_models = {
  'lr' : GridSearchCV(
    LogisticRegression(random_state=42), lr_params,
    cv=5, scoring='f1_weighted', n_jobs=-1
  ),

  'rf' : GridSearchCV(
  RandomForestClassifier(random_state=42), rf_params,
  cv=5, scoring='f1_weighted', n_jobs=-1
  ),

  'gb' : GridSearchCV(
  GradientBoostingClassifier(random_state=42), gb_params,
  cv=5, scoring='f1_weighted', n_jobs=-1
  ),
}

for name, model in exp2_models.items():
  if name == 'lr':
    model.fit(X_train_scaled, y_train)
    best_model = model.best_estimator_
    y_pred = best_model.predict(X_test_scaled)
  else:
    model.fit(X_train, y_train)
    best_model = model.best_estimator_
    y_pred = best_model.predict(X_test)
  
  acc = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='weighted')
  


  print(f"<{name}> Best Params : {model.best_params_}")
  print(f"<{name}> Accuracy Score : {acc:.4f}")
  print(f"<{name}> F1 Score : {f1:.4f}")
  print()

c:\Users\hjjoo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


<lr> Best Params : {'C': 10, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
<lr> Accuracy Score : 0.4573
<lr> F1 Score : 0.4197

<rf> Best Params : {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
<rf> Accuracy Score : 0.5802
<rf> F1 Score : 0.5809

<gb> Best Params : {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.7}
<gb> Accuracy Score : 0.5666
<gb> F1 Score : 0.5655



In [58]:
# 피처 중요도
print("===== EXP2 RF =====")
print(pd.Series(
    exp2_models['rf'].best_estimator_.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False))

print("\n===== EXP2 GB =====")
print(pd.Series(
    exp2_models['gb'].best_estimator_.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False))

===== EXP2 RF =====
rating_count           0.382467
actual_price           0.324699
discount_percentage    0.292833
dtype: float64

===== EXP2 GB =====
rating_count           0.446155
actual_price           0.309117
discount_percentage    0.244727
dtype: float64


In [52]:
# =========================
# 실험 3. lr만 스케일링 + category_encoded 피처 다시 불러오기
# =========================
df2 = pd.read_csv("../data/processed/processed_data_exp3.csv")
df2.head(10)

,actual_price,discount_percentage,rating,rating_count,review_title,review_content,rating_area,category_encoded
0,1099.0,64.0,4.2,24269,"Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,1,1
1,349.0,43.0,4.0,43994,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,0,1
2,1899.0,90.0,3.9,7928,"Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",0,1
3,699.0,53.0,4.2,94363,"Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",1,1
4,399.0,61.0,4.2,16905,"As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",1,1
5,1000.0,85.0,3.9,24871,"It's pretty good,Average quality,very good and...","It's a good product.,Like,Very good item stron...",0,1
6,499.0,65.0,4.1,15188,"Long durable.,good,Does not charge Lenovo m8 t...",Build quality is good and it is comes with 2 y...,1,1
7,299.0,23.0,4.3,30411,"Worth for money - suitable for Android auto,Go...",Worth for money - suitable for Android auto......,2,1
8,999.0,50.0,4.2,179691,Works on linux for me. Get the model with ante...,I use this to connect an old PC to internet. I...,1,1
9,299.0,33.0,4.0,43994,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,0,1


In [59]:
# 피처 구분 - ML X/y, LLM 입력
X2 = df2[["actual_price", "discount_percentage", "rating_count", "category_encoded"]]
y2 = df2["rating_area"]

In [60]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

In [61]:
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

exp3_models = {
  'lr' : LogisticRegression(random_state=42, max_iter=1000),
  'rf' : RandomForestClassifier(random_state=42),
  'gb' : GradientBoostingClassifier(random_state=42)
}

for name, model in exp3_models.items():
  if name == 'lr':
    model.fit(X2_train_scaled, y2_train)
    y2_pred = model.predict(X2_test_scaled)
  else:
    model.fit(X2_train, y2_train)
    y2_pred = model.predict(X2_test)
  
  acc = accuracy_score(y2_test, y2_pred)
  f1 = f1_score(y2_test, y2_pred, average='weighted')
  print(f"<{name}> Accuracy score : {acc:.4f}")
  print(f"<{name}> F1 score : {f1:.4f}")
  print()

<lr> Accuracy score : 0.4573
<lr> F1 score : 0.4172

<rf> Accuracy score : 0.5973
<rf> F1 score : 0.5962

<gb> Accuracy score : 0.5597
<gb> F1 score : 0.5546



In [62]:
# =========================
# 실험 4. lr만 스케일링 + category_encoded 피처 다시 불러오기 + GridSearchCV
# =========================

exp4_models = {
  'lr' : GridSearchCV(
    LogisticRegression(random_state=42), lr_params,
    cv=5, scoring='f1_weighted', n_jobs=-1
  ),

  'rf' : GridSearchCV(
  RandomForestClassifier(random_state=42), rf_params,
  cv=5, scoring='f1_weighted', n_jobs=-1
  ),

  'gb' : GridSearchCV(
  GradientBoostingClassifier(random_state=42), gb_params,
  cv=5, scoring='f1_weighted', n_jobs=-1
  ),
}

for name, model in exp4_models.items():
  if name == 'lr':
    model.fit(X2_train_scaled, y2_train)
    best_model = model.best_estimator_
    y2_pred = best_model.predict(X2_test_scaled)
  else:
    model.fit(X2_train, y2_train)
    best_model = model.best_estimator_
    y2_pred = best_model.predict(X2_test)
  
  acc = accuracy_score(y2_test, y2_pred)
  f1 = f1_score(y2_test, y2_pred, average='weighted')
  print(f"<{name}> Best Params : {model.best_params_}")
  print(f"<{name}> Accuracy score : {acc:.4f}")
  print(f"<{name}> F1 score : {f1:.4f}")
  print()

c:\Users\hjjoo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


<lr> Best Params : {'C': 10, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'saga'}
<lr> Accuracy score : 0.4573
<lr> F1 score : 0.4172

<rf> Best Params : {'max_depth': None, 'max_features': None, 'min_samples_split': 5, 'n_estimators': 100}
<rf> Accuracy score : 0.6246
<rf> F1 score : 0.6249

<gb> Best Params : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.7}
<gb> Accuracy score : 0.6041
<gb> F1 score : 0.6032



In [63]:
# 피처 중요도
print("===== EXP4 RF =====")
print(pd.Series(
    exp4_models['rf'].best_estimator_.feature_importances_,
    index=X2_train.columns
).sort_values(ascending=False))

print("\n===== EXP4 GB =====")
print(pd.Series(
    exp4_models['gb'].best_estimator_.feature_importances_,
    index=X2_train.columns
).sort_values(ascending=False))

===== EXP4 RF =====
rating_count           0.350828
actual_price           0.303278
discount_percentage    0.279008
category_encoded       0.066887
dtype: float64

===== EXP4 GB =====
rating_count           0.418155
actual_price           0.284125
discount_percentage    0.230741
category_encoded       0.066979
dtype: float64
